In [15]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import TimeoutException, StaleElementReferenceException
import time
import pandas as pd
from bs4 import BeautifulSoup

# ================= 浏览器配置 =================
chrome_options = Options()
# chrome_options.add_argument("--headless")
chrome_options.add_argument("--disable-blink-features=AutomationControlled")
chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
chrome_options.add_experimental_option("useAutomationExtension", False)

driver = webdriver.Chrome(options=chrome_options)
driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
wait = WebDriverWait(driver, 10)

data_list = []
base_url = "https://dxy.com/questions"

try:
    print("打开主页...")
    driver.get(base_url)
    time.sleep(3)

    # ================= 获取科室列表 =================
    dept_box = wait.until(
        EC.presence_of_element_located((By.CLASS_NAME, "section-filter-box-border"))
    )

    dept_buttons = dept_box.find_elements(By.CSS_SELECTOR, ".section-filter-box-wrap.normal .tag-button")

    dept_list = []
    for btn in dept_buttons:
        try:
            dept_name = btn.text.strip()
            dept_url = btn.get_attribute("href")
            if dept_name and dept_url:
                dept_list.append((dept_name, dept_url))
        except:
            continue

    print(f"发现 {len(dept_list)} 个科室")

    # ================= 科室循环 =================
    for dept_name, dept_url in dept_list:
        print(f"\n================ 科室: {dept_name} ================")

        driver.get(dept_url)
        time.sleep(3)

        # ================= 获取疾病 =================
        try:
            disease_filter_box = wait.until(
                EC.presence_of_element_located((By.CLASS_NAME, "questions-list-filter-tags"))
            )

            buttons = disease_filter_box.find_elements(
                By.CSS_SELECTOR, ".section-filter-box-wrap.normal .tag-button"
            )

            disease_names = [b.text.strip() for b in buttons if b.text.strip() and b.text.strip() != "全部"]
            print(f"{dept_name} 下发现 {len(disease_names)} 个疾病")

        except:
            print("该科室无疾病分类")
            disease_names = ["全部"]

        # ================= 疾病循环 =================
        for disease_name in disease_names:
            print(f"\n===== {dept_name} | 疾病: {disease_name} =====")

            driver.get(dept_url)
            time.sleep(2)

            # 点击疾病
            if disease_name != "全部":
                try:
                    btn = wait.until(EC.element_to_be_clickable((
                        By.XPATH,
                        f"//div[@class='section-filter-box-wrap normal']//div[@class='tag-button' and contains(text(), '{disease_name}')]"
                    )))
                    driver.execute_script("arguments[0].click();", btn)
                    time.sleep(2)
                except TimeoutException:
                    print("疾病点击失败")
                    continue

            # ================= 分页循环 =================
            max_pages = 10

            for page in range(1, max_pages + 1):
                print(f"{dept_name}-{disease_name} 第 {page} 页")

                try:
                    wait.until(EC.presence_of_element_located(
                        (By.CSS_SELECTOR, "a.question-card.common-card-link")
                    ))
                except:
                    print("无数据")
                    break

                cards = driver.find_elements(By.CSS_SELECTOR, "a.question-card.common-card-link")

                detail_urls = []
                for card in cards:
                    try:
                        url = card.get_attribute("href")
                        if url:
                            detail_urls.append(url)
                    except StaleElementReferenceException:
                        continue

                print(f"本页 {len(detail_urls)} 条")

                # ================= 详情页 =================
                for detail_url in detail_urls:
                    try:
                        driver.get(detail_url)

                        wait.until(EC.presence_of_element_located((By.CLASS_NAME, "dialogs")))
                        time.sleep(1)

                        soup = BeautifulSoup(driver.page_source, "html.parser")

                        # ===== 对话解析 =====
                        dialogs_container = soup.find("div", class_="dialogs")
                        full_dialog = []
                        dialog_time = ""

                        if dialogs_container:
                            time_tag = dialogs_container.find("div", class_="dialog-time")
                            if time_tag:
                                dialog_time = time_tag.get_text(strip=True)

                            dialog_items = dialogs_container.find_all("div", class_="dialog")

                            for dialog in dialog_items:
                                content_div = dialog.find("div", class_="dialog-content")
                                if not content_div:
                                    continue

                                text = content_div.get_text(separator="\n", strip=True)
                                class_list = dialog.get("class", [])

                                if "theme-dark" in class_list:
                                    full_dialog.append(f"【患者】{text}")
                                elif "theme-white" in class_list:
                                    full_dialog.append(f"【医生】{text}")

                        dialog_full_text = "\n\n".join(full_dialog)

                        # ===== 医生信息 =====
                        doctor_info_text = "未获取医生信息"
                        doctor_card = soup.find("div", class_="search-doctor-card")

                        if doctor_card:
                            try:
                                name = doctor_card.find("div", class_="doctor-name").get_text(strip=True)
                                body_contents = doctor_card.find_all("div", class_="doctor-body-content")
                                title = body_contents[0].get_text(strip=True) if len(body_contents) > 0 else ""
                                dept = body_contents[1].get_text(strip=True) if len(body_contents) > 1 else ""
                                hospital = body_contents[2].get_text(strip=True) if len(body_contents) > 2 else ""

                                score_tag = doctor_card.find("div", class_="answer-gold")
                                score = score_tag.get_text(strip=True) if score_tag else ""

                                reply_tag = doctor_card.find("div", class_="reply-count")
                                reply = reply_tag.get_text(strip=True) if reply_tag else ""

                                doctor_info_text = f"{name} | {title} | {dept} | {hospital} | 评分:{score} | {reply}"
                            except:
                                pass

                        data_list.append({
                            "科室": dept_name,
                            "疾病": disease_name,
                            "对话时间": dialog_time,
                            "医生信息": doctor_info_text,
                            "对话全文": dialog_full_text,
                            "链接": detail_url
                        })

                        driver.back()
                        wait.until(EC.presence_of_element_located(
                            (By.CSS_SELECTOR, "a.question-card.common-card-link")
                        ))

                    except Exception as e:
                        print("详情失败:", e)
                        driver.get(dept_url)
                        break

                # ===== 翻页 =====
                if page == max_pages:
                    break

                try:
                    next_page_btn = wait.until(EC.element_to_be_clickable((
                        By.XPATH,
                        f"//div[@class='pagination-item' and text()='{page + 1}']"
                    )))
                    driver.execute_script("arguments[0].click();", next_page_btn)
                    time.sleep(2)
                except:
                    print("没有更多页")
                    break

except Exception as e:
    print("程序异常:", e)

finally:
    driver.quit()
    print("浏览器关闭")

# ================= 保存Excel =================
if data_list:
    df = pd.DataFrame(data_list)
    filename = "丁香医生_全科室.xlsx"
    df.to_excel(filename, index=False, engine="openpyxl")
    print(f"\n成功抓取 {len(data_list)} 条数据，已保存至 {filename}")
else:
    print("未抓取到数据")

打开主页...
发现 33 个科室

================ 科室: 皮肤科 ================
皮肤科 下发现 11 个疾病

===== 皮肤科 | 疾病: 痤疮 =====
皮肤科-痤疮 第 1 页
本页 10 条
皮肤科-痤疮 第 2 页
本页 10 条
皮肤科-痤疮 第 3 页
本页 10 条
皮肤科-痤疮 第 4 页
本页 10 条
皮肤科-痤疮 第 5 页
本页 10 条
皮肤科-痤疮 第 6 页
本页 10 条
皮肤科-痤疮 第 7 页
本页 10 条
皮肤科-痤疮 第 8 页
本页 10 条
皮肤科-痤疮 第 9 页
本页 10 条
皮肤科-痤疮 第 10 页
本页 10 条

===== 皮肤科 | 疾病: 荨麻疹 =====
皮肤科-荨麻疹 第 1 页
本页 10 条
皮肤科-荨麻疹 第 2 页
本页 10 条
皮肤科-荨麻疹 第 3 页
本页 10 条
皮肤科-荨麻疹 第 4 页
本页 10 条
皮肤科-荨麻疹 第 5 页
本页 10 条
皮肤科-荨麻疹 第 6 页
本页 10 条
皮肤科-荨麻疹 第 7 页
本页 10 条
皮肤科-荨麻疹 第 8 页
本页 10 条
皮肤科-荨麻疹 第 9 页
本页 10 条
皮肤科-荨麻疹 第 10 页
本页 10 条

===== 皮肤科 | 疾病: 瘙痒症 =====
皮肤科-瘙痒症 第 1 页
本页 10 条
皮肤科-瘙痒症 第 2 页
本页 10 条
皮肤科-瘙痒症 第 3 页
本页 10 条
皮肤科-瘙痒症 第 4 页
本页 10 条
皮肤科-瘙痒症 第 5 页
本页 10 条
皮肤科-瘙痒症 第 6 页
本页 10 条
皮肤科-瘙痒症 第 7 页
本页 10 条
皮肤科-瘙痒症 第 8 页
本页 10 条
皮肤科-瘙痒症 第 9 页
本页 10 条
皮肤科-瘙痒症 第 10 页
本页 10 条

===== 皮肤科 | 疾病: 特应性皮炎 =====
皮肤科-特应性皮炎 第 1 页
本页 10 条
皮肤科-特应性皮炎 第 2 页
本页 10 条
皮肤科-特应性皮炎 第 3 页
本页 10 条
皮肤科-特应性皮炎 第 4 页
本页 10 条
皮肤科-特应性皮炎 第 5 页
本页 10 条
皮肤科-特应性皮炎 第 6 页
本页 10 条
皮肤科-特应性皮炎 第 7 页
本页

In [16]:
data = pd.read_excel("丁香医生_全科室.xlsx",header=0)
data

,科室,疾病,对话时间,医生信息,对话全文,链接
0,皮肤科,痤疮,2026年02月11日,马硕 | 副主任医师 | 皮肤科 | 沈阳市第七人民医院 | 评分:4.98分 | 月回答423,【患者】症状及患病时长：大概三个月了\n\n就医及用药情况：没有就医没有用药\n\n需要解答...,https://dxy.com/question/112547344
1,皮肤科,痤疮,2026年02月10日,马硕 | 副主任医师 | 皮肤科 | 沈阳市第七人民医院 | 评分:4.98分 | 月回答423,【患者】症状及患病时长：本人目前25周岁，脸上额头，脸颊两侧，背部会出白色脓包痘痘，脸部红色...,https://dxy.com/question/111791234
2,皮肤科,痤疮,2026年02月09日,邓琳 | 副主任医师 | 皮肤科 | 杭州市第一人民医院 | 评分:4.92分 | 月回答920,【患者】症状及患病时长：\n一年\n就医及用药情况：\n无\n需要解答的问题：\n男孩15....,https://dxy.com/question/112147602
3,皮肤科,痤疮,2026年02月07日,邓琳 | 副主任医师 | 皮肤科 | 杭州市第一人民医院 | 评分:4.92分 | 月回答920,【患者】症状及患病时长：5年\n\n就医及用药情况：左侧太阳穴附近，总是出现红肿型痘痘，容易...,https://dxy.com/question/112736924
4,皮肤科,痤疮,2026年02月03日,兰婷 | 主治医师 | 皮肤科 | 西安市第八医院 | 评分:4.92分 | 月回答744,【患者】症状及患病时长：额头和发际线边缘长了一些痘痘，也有一些痘印，脸颊两侧毛孔粗大，皮肤摸...,https://dxy.com/question/112136638
...,...,...,...,...,...,...
13519,疫苗科,感冒,2020年09月08日,钟文映 | 副主任医师 | 免疫规划科 | 佛山复星禅诚医院 | 评分:5.00分 | 月回答3,【患者】您好，医生我昨天下午两点钟接种了宫颈癌疫苗请问今天可以做瑜伽洗澡吗\n\n【医生】您...,https://dxy.com/question/68786802
13520,疫苗科,肺炎链球菌性肺炎,2020年03月11日,田伟 | 副主任医师 | 免疫规划科 | 安徽某疾控中心 | 评分:4.89分 | 月回答29,【患者】我儿子12月26日出生，因为疫情到2月19日才打上满月针乙肝和卡介苗，今天3月11日...,https://dxy.com/question/49372439
13521,疫苗科,肺炎链球菌性肺炎,2020年03月09日,田伟 | 副主任医师 | 免疫规划科 | 安徽某疾控中心 | 评分:4.89分 | 月回答29,【患者】医生你好，我家宝宝19年12-12出生，目前只有出生接种卡介苗和乙肝疫苗，以及一个月...,https://dxy.com/question/49179635
13522,疫苗科,肺炎链球菌性肺炎,2020年01月27日,张皓 | 主治医师 | 免疫规划科 | 深圳市罗湖区人民医院 | 评分:1.00分 | 月回答7,【患者】您好！我长年居住武汉，1.19日返乡。1.25日出现了咳嗽的症状，随后吃了头孢胶囊，...,https://dxy.com/question/42400583


In [18]:
data_uni = data.drop_duplicates(subset="链接")
data_uni

,科室,疾病,对话时间,医生信息,对话全文,链接
0,皮肤科,痤疮,2026年02月11日,马硕 | 副主任医师 | 皮肤科 | 沈阳市第七人民医院 | 评分:4.98分 | 月回答423,【患者】症状及患病时长：大概三个月了\n\n就医及用药情况：没有就医没有用药\n\n需要解答...,https://dxy.com/question/112547344
1,皮肤科,痤疮,2026年02月10日,马硕 | 副主任医师 | 皮肤科 | 沈阳市第七人民医院 | 评分:4.98分 | 月回答423,【患者】症状及患病时长：本人目前25周岁，脸上额头，脸颊两侧，背部会出白色脓包痘痘，脸部红色...,https://dxy.com/question/111791234
2,皮肤科,痤疮,2026年02月09日,邓琳 | 副主任医师 | 皮肤科 | 杭州市第一人民医院 | 评分:4.92分 | 月回答920,【患者】症状及患病时长：\n一年\n就医及用药情况：\n无\n需要解答的问题：\n男孩15....,https://dxy.com/question/112147602
3,皮肤科,痤疮,2026年02月07日,邓琳 | 副主任医师 | 皮肤科 | 杭州市第一人民医院 | 评分:4.92分 | 月回答920,【患者】症状及患病时长：5年\n\n就医及用药情况：左侧太阳穴附近，总是出现红肿型痘痘，容易...,https://dxy.com/question/112736924
4,皮肤科,痤疮,2026年02月03日,兰婷 | 主治医师 | 皮肤科 | 西安市第八医院 | 评分:4.92分 | 月回答744,【患者】症状及患病时长：额头和发际线边缘长了一些痘痘，也有一些痘印，脸颊两侧毛孔粗大，皮肤摸...,https://dxy.com/question/112136638
...,...,...,...,...,...,...
13519,疫苗科,感冒,2020年09月08日,钟文映 | 副主任医师 | 免疫规划科 | 佛山复星禅诚医院 | 评分:5.00分 | 月回答3,【患者】您好，医生我昨天下午两点钟接种了宫颈癌疫苗请问今天可以做瑜伽洗澡吗\n\n【医生】您...,https://dxy.com/question/68786802
13520,疫苗科,肺炎链球菌性肺炎,2020年03月11日,田伟 | 副主任医师 | 免疫规划科 | 安徽某疾控中心 | 评分:4.89分 | 月回答29,【患者】我儿子12月26日出生，因为疫情到2月19日才打上满月针乙肝和卡介苗，今天3月11日...,https://dxy.com/question/49372439
13521,疫苗科,肺炎链球菌性肺炎,2020年03月09日,田伟 | 副主任医师 | 免疫规划科 | 安徽某疾控中心 | 评分:4.89分 | 月回答29,【患者】医生你好，我家宝宝19年12-12出生，目前只有出生接种卡介苗和乙肝疫苗，以及一个月...,https://dxy.com/question/49179635
13522,疫苗科,肺炎链球菌性肺炎,2020年01月27日,张皓 | 主治医师 | 免疫规划科 | 深圳市罗湖区人民医院 | 评分:1.00分 | 月回答7,【患者】您好！我长年居住武汉，1.19日返乡。1.25日出现了咳嗽的症状，随后吃了头孢胶囊，...,https://dxy.com/question/42400583


In [19]:
data_uni.to_excel("丁香医生_全科室_去重.xlsx",index=False)